In [ ]:
# TODO: add multi layer support

CONFIG = {
    "block_size" : 64,
    "n_embed" : 128,
    "batch_size" : 64,
    "hidden_size" : 256,  # Increased from 10 to 256 for better capacity
    "n_epochs" : 50,      # Increased from 10 to 50    "batch_size" : 64,
    "learning_rate" : 3e-3,  # Increased from 1e-4 to 3e-3
}

In [ ]:
import torch
DEVICE = torch.device("mps")

In [ ]:
# load shakespear
with open("data/lotr.txt", "r", encoding="utf-8") as f: DATASET_TEXT = f.read()
DATASET_TEXT[:10000]

In [ ]:
# create tokenizer
VOCAB = sorted(list(set(DATASET_TEXT)))
ctoi = dict([(c, i) for i, c in enumerate(VOCAB)])
itoc = dict([(i, c) for i, c in enumerate(VOCAB)])
encode = lambda str: [ctoi[c] for c in str]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
len(VOCAB), decode(encode("hello world"))

In [ ]:
# tokenize shakespear
DATASET_TOKENS = encode(DATASET_TEXT)
len(DATASET_TOKENS), DATASET_TOKENS[:10]

In [ ]:
import torch
from torch.utils.data import Dataset

class LOTRDataset(Dataset):
    def __init__(self, tokens, block_size=10):
        super().__init__()
        tokens = torch.tensor(tokens)
        chunk_size = block_size + 1
        n_chunks = len(tokens) // chunk_size
        chunks = tokens[:chunk_size * n_chunks]
        chunks = chunks.view(n_chunks, chunk_size)
        self.chunks = chunks
    
    def __getitem__(self, idx):
        tokens = self.chunks[idx]
        x = tokens[:-1]
        y = tokens[1:]
        return x, y
        
    def __len__(self):
        return len(self.chunks)

# Fixed: Pass block_size from CONFIG instead of using default 10
DATASET = LOTRDataset(DATASET_TOKENS, block_size=CONFIG["block_size"])
for i in range(10):
    x, y = DATASET[i]
    print(decode(x.tolist()))
    print(decode(y.tolist()))
    print("---")

In [ ]:
from torch.utils.data import DataLoader

dataloader = DataLoader(
    DATASET,
    batch_size=CONFIG["batch_size"],
    shuffle=True
)
print(len(DATASET))
for x,y in dataloader:
    for i in range(x.shape[0]):
        print(decode(x[i].tolist()))
        print(decode(y[i].tolist()))
        print("----")
        break

In [ ]:
import torch.nn as nn

class RNNCell(nn.Module):
    def __init__(self, hidden_size=CONFIG["hidden_size"], n_embed=CONFIG["n_embed"]):
        super().__init__()
        self.n_embed = n_embed
        self.hidden_size = hidden_size

        self.Wx = nn.Linear(n_embed, hidden_size, bias=True)
        self.Wh = nn.Linear(hidden_size, hidden_size, bias=True)
    
    def forward(self, x):
        B, T, C = x.shape

        outputs = []
        n_t = x.shape[1]
        ht = torch.zeros((B, self.hidden_size)).to(DEVICE)

        x_transformed = self.Wx(x)
        
        hidden_states = []
        for t in range(n_t):
            xt_hat = x_transformed[:, t, :]
            ht_hat = self.Wh(ht)
            ht = torch.tanh(xt_hat + ht_hat)
            hidden_states.append(ht)
            outputs.append(ht)
        outputs = torch.stack(outputs, dim=1) # TODO: why the dim=1?
        hidden_states = torch.stack(hidden_states, dim=1)

        return outputs, hidden_states
        
class RNN(nn.Module):
    def __init__(self, vocab_size, hidden_size=CONFIG["hidden_size"], n_embed=CONFIG["n_embed"]):
        super().__init__()
        self.n_embed = n_embed
        self.hidden_size = hidden_size
        self.embeddings = nn.Embedding(vocab_size, n_embed)

        # Replace Wx + Wh + loop with PyTorch's RNN
        #self.rnn = nn.RNN(
        #    input_size=n_embed,      # Takes embedding dimension as input
        #    hidden_size=hidden_size,  # Outputs hidden_size dimension
        #    num_layers=1,             # Single layer
        #    batch_first=True,         # Input/output: (batch, seq, feature)
        #    nonlinearity='tanh'       # Same activation as your implementation
        #)
        self.rnn = RNNCell(
            n_embed=n_embed,
            hidden_size=hidden_size,
        )

        self.ffn = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        x_emb = self.embeddings(x)
        outputs, _ = self.rnn(x_emb)
        logits = self.ffn(outputs)
        return logits
        

VOCAB_SIZE = len(ctoi)
x, y = next(iter(dataloader))
x = x.to(DEVICE)
model = RNN(VOCAB_SIZE).to(DEVICE)
model(x)

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

In [ ]:
total = 0
for name, param in model.named_parameters():
    total += param.numel()
    print(name, param.numel())
print(total)

In [ ]:
import torch.nn.functional as F

# TODO: must keep context window cropped to block size
# TODO: max len must take promtp into account
# TODO: this can be sped up
def generate(model, prompt, block_size=CONFIG["block_size"], max_len=None):
    print(prompt, end="")
    tokens = encode(prompt)

    if max_len is None: max_len = block_size - len(tokens)
    tokens = torch.tensor(tokens, device=DEVICE).unsqueeze(0)
    for _ in range(max_len):
        logits = model(tokens).squeeze(0)
        probs = F.softmax(logits, dim=-1)
        idxs = torch.multinomial(probs, num_samples=1)
        token = idxs[-1, :]
        token = token.item()
        print(decode([token]), end="")
        token_t = torch.tensor(token, device=DEVICE).unsqueeze(0).unsqueeze(0)
        tokens = torch.cat([tokens, token_t], dim=-1)
generate(model, "Frodo picked up")

In [ ]:
model = RNN(VOCAB_SIZE)
model = model.to(DEVICE)
model

In [ ]:
from tqdm import tqdm
from torch.nn.utils import clip_grad_norm_

lr = CONFIG["learning_rate"]
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
losses = []
gradnorms = []
leave = False
for epoch in range(CONFIG["n_epochs"]):
    bar = tqdm(dataloader) # TODO: reuse bar
    if leave: break
    for x, y in bar:
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = model(x)
        #logits = F.softmax(logits, dim=-1)
        #y_pred = torch.multinomial(logits, dim=-1, num_samples=1)
        logits = logits.permute(0, 2, 1) # -> [N, C, L]      (64, 108, 10)
        loss = F.cross_entropy(logits, y)
        optimizer.zero_grad()
        loss.backward()

        total_norm = clip_grad_norm_(model.parameters(), max_norm=float('inf'))
        gradnorms.append(total_norm.item())

        optimizer.step()
        loss_i = loss.item()
        bar.set_postfix(dict(epoch=epoch, loss=loss_i))
        losses.append(loss_i)
        #print(loss)


In [ ]:
from matplotlib import pyplot as plt

fig, ax = plt.subplots(1, 2)

ax[0].plot(losses)
ax[1].plot(gradnorms)


In [ ]:
generate(model, "frodo", max_len=1024)